In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import yaml
import joblib
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

print('Libraries loaded.')

## 1. Load Configuration

In [ ]:
with open('ml_config.yaml', 'r') as f:
    ml_cfg = yaml.safe_load(f)

FEATURE_COLS = ml_cfg['features']
MODEL_PARAMS = ml_cfg['model']['params']
OUTPUT_PATH  = ml_cfg['training']['output_path']
MIN_SAMPLES  = ml_cfg['training']['min_samples']

print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')
print(f'Model params: {MODEL_PARAMS}')

## 2. Load Data from Database

In [ ]:
DB_URL = 'sqlite:///../netscan.db'
engine = create_engine(DB_URL)

df = pd.read_sql_table('network_features', engine)
print(f'Loaded {len(df)} rows from network_features')
df.head()

## 3. Explore & Visualise

In [ ]:
df[FEATURE_COLS].describe().T

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
for ax, col in zip(axes.flat, FEATURE_COLS):
    ax.hist(df[col].dropna(), bins=30, edgecolor='black', alpha=0.7)
    ax.set_title(col, fontsize=9)
plt.tight_layout()
plt.suptitle('Feature Distributions', y=1.02, fontsize=14)
plt.show()

In [ ]:
corr = df[FEATURE_COLS].corr()
plt.figure(figsize=(10, 8))
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar()
plt.xticks(range(len(FEATURE_COLS)), FEATURE_COLS, rotation=90, fontsize=8)
plt.yticks(range(len(FEATURE_COLS)), FEATURE_COLS, fontsize=8)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 4. Preprocess

In [ ]:
X = df[FEATURE_COLS].fillna(0).values.astype(float)
print(f'Feature matrix shape: {X.shape}')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Scaling complete.')

## 5. Train IsolationForest

In [ ]:
if len(X_scaled) < MIN_SAMPLES:
    print(f'⚠ Only {len(X_scaled)} samples — consider capturing more '
          f'baseline traffic (need ≥{MIN_SAMPLES}).')

model = IsolationForest(**MODEL_PARAMS)
model.fit(X_scaled)
print('Training complete.')

## 6. Evaluate

In [ ]:
scores = model.score_samples(X_scaled)
df['anomaly_score'] = scores
df['is_anomaly'] = model.predict(X_scaled) == -1

print(f'Anomalies detected: {df["is_anomaly"].sum()} / {len(df)}')

plt.figure(figsize=(10, 4))
plt.hist(scores, bins=50, edgecolor='black', alpha=0.7)
plt.axvline(x=scores[df['is_anomaly']].max(), color='red', ls='--', label='anomaly threshold')
plt.xlabel('Anomaly Score')
plt.ylabel('Count')
plt.title('IsolationForest Anomaly Score Distribution')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Show top anomalies
cols_display = ['src_ip', 'window_start', 'anomaly_score', 'is_anomaly'] + FEATURE_COLS[:6]
df.nsmallest(10, 'anomaly_score')[cols_display]

## 7. Save Model

In [ ]:
# Save the model + scaler as a pipeline-like dict
artifact = {
    'model': model,
    'scaler': scaler,
    'feature_cols': FEATURE_COLS,
}
joblib.dump(model, OUTPUT_PATH)
print(f'Model saved to {OUTPUT_PATH}')

## 8. Test on Sample Vectors

Score the example scenarios from the project spec.

In [ ]:
samples = {
    'VPN (OpenVPN)': [45, 3, 1, 8e6, 25e6, 1300, 50, 0.01, 0.002, 0, 45, 1, 1, 1194, 0.9, 0.0],
    'Gambling':      [80, 15, 6, 1e6, 7e6, 700, 300, 0.05, 0.03, 80, 0, 20, 2, 443, 0.0, 0.7],
    'Torrent':       [200, 60, 2, 15e6, 40e6, 1000, 300, 0.005, 0.003, 120, 80, 5, 20, 6881, 0.1, 0.3],
    'Normal':        [12, 5, 4, 2e5, 1.5e6, 600, 250, 0.2, 0.15, 12, 0, 4, 2, 443, 0.0, 0.0],
}

for label, vec in samples.items():
    x = np.array([vec], dtype=float)
    x_sc = scaler.transform(x)
    score = model.score_samples(x_sc)[0]
    pred  = model.predict(x_sc)[0]
    tag   = '🔴 ANOMALY' if pred == -1 else '🟢 normal'
    print(f'{label:20s}  score={score:.4f}  {tag}')